# 柱 A：条件付け × 目的関数 × 学習手順（CheXpert）

hypernetwork（Spatial LoRA）で属性に条件付けると、同じ目的関数・同じ学習手順の ResNet-50 より、全体の性能を落とさずに
最悪群を上げられるかを見る。格子と条件の決め方は [README](README.md)、run は [runs.md](runs.md)。

- **主の評価**：固定しきい値 0.5 での群別 TPR・FPR の差（Eopp1・Eopp0・Eodds）と、最悪群の FPR（No Finding を陽性とするので、
  FPR は所見のある人を正常と判定する見落とし）。群は年齢（65 歳）× race の 8 群、率のセルの下限は 20 件。
- **副の評価**：全体 AUROC、最悪群 AUROC、AUROC gap。
- **改善と呼ぶ条件**：全体 AUROC が有意に下がらず、Eodds と最悪群の FPR がともに有意に下がること。
- **checkpoint**：主は最終 epoch（全手法に同じ規則）、副は val AUROC が最大の epoch。
- **区間**：test の患者単位 bootstrap 95%（200 回）。3 seed の平均どうしの差で、seed の揺れは含まない。

入力は `compute.py` が書いた `results/run_metrics_test.csv` と `cache/bootstrap_test.npz`（予測 cache は ws11 で
`analysis/common/predictions.py` が書いたもの）。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import STYLE_SHEET, study_dir  # noqa: E402
from analysis.hypernet_conditioning_grid.comparisons import (  # noqa: E402
    LABELS,
    Bootstrap,
    compare,
    condition_means,
    improved,
)

plt.style.use(STYLE_SHEET)
PACKAGE = study_dir("hypernet_conditioning_grid")
points = pd.read_csv(PACKAGE / "results" / "run_metrics_test.csv")
means = condition_means(points)
bootstrap = Bootstrap(PACKAGE / "cache" / "bootstrap_test.npz", points)
MODELS = ["resnet", "spatial_lora", "spatial_lora_fc"]
OBJECTIVES = ["erm", "group_dro", "eo_rate"]
COLORS = {"erm": "tab:gray", "group_dro": "tab:blue", "eo_rate": "tab:red"}
MARKERS = {"resnet": "o", "spatial_lora": "s", "spatial_lora_fc": "^"}
SHOWN = {"auroc": "overall AUROC", "eodds": "Eodds", "worst_fpr": "worst-group FPR", "min_tpr": "worst-group TPR",
         "worst_group_auroc": "worst-group AUROC"}
print(f"{len(points)} rows, {points.run_id.nunique()} runs, seeds per condition: {sorted(means.seeds.unique())}")
print("groups evaluated (TPR / FPR):", sorted(points.groups_tpr.unique()), sorted(points.groups_fpr.unique()))

## 1. 全体の性能と群間差の位置

各条件の 3 seed 平均を、横軸 Eodds（小さいほど群間差が小さい）、縦軸を全体 AUROC と最悪群 FPR に置く。
改善は「左へ動き、AUROC は下がらず、最悪群 FPR は下がる」方向。塗りつぶしが最終 epoch、白抜きが best val AUROC の epoch。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
for column, procedure in enumerate(["one_stage", "two_stage"]):
    subset = means[means.procedure == procedure]
    parent = means[(means.procedure == "one_stage") & (means.model == "resnet") & (means.objective == "erm")]
    for row, metric in enumerate(["auroc", "worst_fpr"]):
        ax = axes[row, column]
        for item in subset.itertuples():
            filled = item.checkpoint == "last"
            ax.errorbar(item.eodds, getattr(item, metric), xerr=item.eodds_sd, yerr=getattr(item, f"{metric}_sd"),
                        marker=MARKERS[item.model], color=COLORS[item.objective], mfc=COLORS[item.objective] if filled else "white",
                        ms=8, lw=0.8, capsize=2)
        if procedure == "two_stage":
            best_parent = parent[parent.checkpoint == "best"].iloc[0]
            ax.plot(best_parent.eodds, best_parent[metric], marker="*", color="black", ms=14, ls="none")
        ax.set_ylabel(SHOWN[metric])
        ax.set_title(LABELS[procedure])
for ax in axes[1]:
    ax.set_xlabel("Eodds (age × race, threshold 0.5)")
handles = [plt.Line2D([], [], color=COLORS[o], marker="o", ls="none", label=LABELS[o]) for o in OBJECTIVES]
handles += [plt.Line2D([], [], color="black", marker=MARKERS[m], ls="none", mfc="white", label=LABELS[m]) for m in MODELS]
handles += [plt.Line2D([], [], color="black", marker="*", ms=12, ls="none", label="parent ResNet ERM (best epoch)")]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False)
fig.suptitle("CheXpert No Finding, test, mean ± SD over 3 seeds (filled: last epoch, open: best val AUROC epoch)")
fig.tight_layout(rect=(0, 0.08, 1, 0.96))

## 2. 目的関数の効果：同じモデル・同じ手順の ERM との差

GroupDRO・EO rate を、同じモデル・同じ学習手順・同じ checkpoint 規則の ERM と比べる。区間が 0 をまたがなければ有意。
AUROC は上が良い、Eodds と最悪群 FPR は下が良い、最悪群 TPR は上が良い。

In [ ]:
def forest(pairs, title):
    metrics = ["auroc", "eodds", "worst_fpr", "min_tpr"]
    fig, axes = plt.subplots(1, len(metrics), figsize=(15, 0.42 * len(pairs) + 1.5), sharey=True)
    for i, (label, a, b, color) in enumerate(pairs):
        diff = compare(means, bootstrap, a, b).set_index("metric")
        for ax, metric in zip(axes, metrics):
            d = diff.loc[metric]
            ax.errorbar(d["diff"], i, xerr=[[d["diff"] - d["low"]], [d["high"] - d["diff"]]], fmt="o", color=color, capsize=2)
    for ax, metric in zip(axes, metrics):
        ax.axvline(0, color="black", lw=0.8)
        ax.set_title(f"Δ {SHOWN[metric]}")
    axes[0].set_yticks(range(len(pairs)), [p[0] for p in pairs])
    axes[0].invert_yaxis()
    fig.suptitle(title)
    fig.tight_layout()

for checkpoint in ["last", "best"]:
    pairs = []
    for procedure in ["one_stage", "two_stage"]:
        for model in MODELS:
            if procedure == "two_stage" and model == "resnet":
                continue
            for objective in ["group_dro", "eo_rate"]:
                a = (procedure, model, objective, checkpoint)
                b = (procedure, model, "erm", checkpoint)
                pairs.append((f"{LABELS[procedure][:7]} | {LABELS[model]} | {LABELS[objective]}", a, b, COLORS[objective]))
    forest(pairs, f"objective − ERM (same model, same procedure), {LABELS[checkpoint]}, 95% patient bootstrap")

## 3. 条件付けの効果：同じ目的関数の ResNet-50 との差

1 stage は、同じ目的関数・同じ checkpoint 規則の ResNet-50 と比べる。2 stage は、凍結した親（ResNet-50 ERM、best epoch）と比べる
（2 stage の ResNet 側の対照は回していない）。

In [ ]:
for checkpoint in ["last", "best"]:
    pairs = []
    for model in ["spatial_lora", "spatial_lora_fc"]:
        for objective in OBJECTIVES:
            a = ("one_stage", model, objective, checkpoint)
            b = ("one_stage", "resnet", objective, checkpoint)
            pairs.append((f"1 stage | {LABELS[model]} − ResNet | {LABELS[objective]}", a, b, COLORS[objective]))
    for model in ["spatial_lora", "spatial_lora_fc"]:
        for objective in OBJECTIVES:
            a = ("two_stage", model, objective, checkpoint)
            b = ("one_stage", "resnet", "erm", "best")
            pairs.append((f"2 stage | {LABELS[model]} | {LABELS[objective]} − parent", a, b, COLORS[objective]))
    forest(pairs, f"conditioned − unconditioned, {LABELS[checkpoint]}, 95% patient bootstrap")

## 4. 「改善と呼ぶ条件」を満たした比較

上の全比較について、改善の 3 条件（AUROC が有意に下がらない・Eodds が有意に下がる・最悪群 FPR が有意に下がる）を判定し、
`results/improvement_test.csv` に残す。

In [ ]:
records = []
for checkpoint in ["last", "best"]:
    for procedure in ["one_stage", "two_stage"]:
        for model in MODELS:
            for objective in OBJECTIVES:
                a = (procedure, model, objective, checkpoint)
                if a not in set(map(tuple, means[["procedure", "model", "objective", "checkpoint"]].to_numpy())):
                    continue
                references = []
                if objective != "erm":
                    references.append(("vs ERM", (procedure, model, "erm", checkpoint)))
                if procedure == "one_stage" and model != "resnet":
                    references.append(("vs ResNet", ("one_stage", "resnet", objective, checkpoint)))
                if procedure == "two_stage":
                    references.append(("vs parent", ("one_stage", "resnet", "erm", "best")))
                for name, b in references:
                    diff = compare(means, bootstrap, a, b).set_index("metric")
                    records.append({"procedure": procedure, "model": model, "objective": objective, "checkpoint": checkpoint,
                                    "reference": name, **{f"d_{m}": diff.loc[m, "diff"] for m in SHOWN}, **improved(diff.reset_index())})
judgement = pd.DataFrame(records)
judgement.to_csv(PACKAGE / "results" / "improvement_test.csv", index=False)
print(f"{judgement.improved.sum()} / {len(judgement)} comparisons meet all three conditions")
for criterion in ["auroc_not_worse", "eodds_smaller", "worst_fpr_lower"]:
    print(f"  {criterion}: {judgement[criterion].sum()} / {len(judgement)}")
for row in judgement[judgement.improved].itertuples():
    print(f"  improved: {LABELS[row.procedure]} | {LABELS[row.model]} | {LABELS[row.objective]} | {LABELS[row.checkpoint]} | {row.reference}")

## まとめ

（結果を見てから書く）